### **Notebook com foco em trazer os dados da camada Bronze, realizar tratamento de dados e salvar em tabelas da camada Silver.**

**Importando as bibliotecas**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Definindo as variáveis que serão utilizadas no notabook:**

In [0]:
path_tabela_viagens = "nyc_taxi_data.bronze.viagens"

path_tabela_zonas = "nyc_taxi_data.bronze.zonas"

**Definindo os dataframes a partir das tabelas da camada bronze:**

In [0]:


df_viagens = spark.table(path_tabela_viagens)

df_zonas = spark.table(path_tabela_zonas)

**Investigando se os dados estão corretos:**

In [0]:
df_viagens.show(10)

df_zonas.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

**Investigando se o schema está correto:**

In [0]:
df_viagens.printSchema()
df_zonas.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nulla

###**Início das transformações de dados:**

**Alterando os nomes das colunas do dataframe de viagens:**

In [0]:
df_viagens = df_viagens.withColumnRenamed('VendorID','Id_Fornecedor_Tecnologia').withColumnRenamed('tpep_pickup_datetime','Inicio_Corrida').withColumnRenamed('tpep_dropoff_datetime','Fim_Corrida').withColumnRenamed('passenger_count','Qtd_Passageiros').withColumnRenamed('trip_distance','Distancia_corrida_milhas').withColumnRenamed('RatecodeID','Id_tarifa').withColumnRenamed('store_and_fwd_flag','Flag_armazenamento').withColumnRenamed('PULocationID','Zona_inicio_corrida').withColumnRenamed('DOLocationID','Zona_fim_corrida').withColumnRenamed('payment_type','Id_tipo_pagamento').withColumnRenamed('fare_amount','Valor_corrida_Tempo_Distancia').withColumnRenamed('extra','Taxas_Diversas').withColumnRenamed('mta_tax','Taxa_MTA').withColumnRenamed('tip_amount','Valor_Gorgetas_Cartao_Credito').withColumnRenamed('tolls_amount','Valor_Pedagios').withColumnRenamed('improvement_surcharge','Taxa_Melhoria').withColumnRenamed('total_amount','Valor_Total_Corrida').withColumnRenamed('congestion_surcharge','Taxa_Congestionamento').withColumnRenamed('Airport_fee','Taxa_Aeroporto').withColumnRenamed('cbd_congestion_fee','Taxa_Congestionamento_CBD')

df_viagens.show(10)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|
+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+---

**Alterando os nomes das colunas do dataframe de zonas da cidade:**

In [0]:
df_zonas = df_zonas.withColumnRenamed('LocationID','Id_Localizacao').withColumnRenamed('Borough','Distrito').withColumnRenamed('Zone','Zona').withColumnRenamed('service_zone','Zona_Servico')

df_zonas.show(10)

+--------------+-------------+--------------------+------------+
|Id_Localizacao|     Distrito|                Zona|Zona_Servico|
+--------------+-------------+--------------------+------------+
|             1|          EWR|      Newark Airport|         EWR|
|             2|       Queens|         Jamaica Bay|   Boro Zone|
|             3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|             4|    Manhattan|       Alphabet City| Yellow Zone|
|             5|Staten Island|       Arden Heights|   Boro Zone|
|             6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|             7|       Queens|             Astoria|   Boro Zone|
|             8|       Queens|        Astoria Park|   Boro Zone|
|             9|       Queens|          Auburndale|   Boro Zone|
|            10|       Queens|        Baisley Park|   Boro Zone|
+--------------+-------------+--------------------+------------+
only showing top 10 rows


**Criando coluna de mês da corrida no dataframe de viagens**

In [0]:
df_viagens = df_viagens.withColumn('Mes_Corrida', month(col("Inicio_Corrida")))

df_viagens.show(10)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|Mes_Corrida|
+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+---------

**Criando uma coluna de mês pela abreviação do mês no dataframe de viagens**

In [0]:
df_viagens = df_viagens.withColumn('Nome_Mes', date_format(col('Inicio_Corrida'),'MMM'))

**Criando coluna do dia da corrida no dataframe de viagens**

In [0]:
df_viagens = df_viagens.withColumn('Dia_Corrida',dayofmonth(col('Inicio_Corrida')))

**Criando uma coluna de data da corrida**

In [0]:
df_viagens = df_viagens.withColumn("Data_Corrida", to_date(col("Inicio_Corrida")))

df_viagens.show(3)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|Mes_Corrida|Nome_Mes|Dia_Corrida|Data_Corrida|
+------------------------+-------------------+-------------------+---------------+------------------------+---------+---------------

**Criando uma coluna de hora da corrida**

In [0]:
df_viagens = df_viagens.withColumn("Hora_Corrida", hour(col("Inicio_Corrida")))

df_viagens.show(3)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|Mes_Corrida|Nome_Mes|Dia_Corrida|Data_Corrida|Hora_Corrida|
+------------------------+-------------------+-------------------+---------------+------------------------

**Criando uma coluna de "períodos" do dia**

In [0]:
df_viagens = df_viagens.withColumn("Periodo_Dia",
    when(col("Hora_Corrida").between(6,11), "Manhã")
    .when(col("Hora_Corrida").between(12,17), "Tarde")
    .when(col("Hora_Corrida").between(18,23), "Noite")
    .otherwise("Madrugada")                                   
)

df_viagens.show(3)


+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|Mes_Corrida|Nome_Mes|Dia_Corrida|Data_Corrida|Hora_Corrida|Periodo_Dia|
+------------------------+-------------------+-------------------+---------------+

**Criando Coluna de Dia da Semana**

In [0]:
df_viagens = df_viagens.withColumn("Dia_Semana", date_format("Inicio_Corrida", "EEEE"))

df_viagens.show(3)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|Mes_Corrida|Nome_Mes|Dia_Corrida|Data_Corrida|Hora_Corrida|Periodo_Dia|Dia_Semana|
+------------------------+-------------------+--------------

**Criando coluna numérica de dia da semana**

In [0]:
df_viagens = df_viagens.withColumn("Numero_Dia_Semana", dayofweek("Inicio_Corrida"))

df_viagens.show(3)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|Mes_Corrida|Nome_Mes|Dia_Corrida|Data_Corrida|Hora_Corrida|Periodo_Dia|Dia_Semana|Numero_Dia_Semana|
+-----------------------

**Calculando a duração total das viagens em segundos no dataframe de viagens**

In [0]:
df_viagens = df_viagens.withColumn('Duracao_Viagem_Segundos', unix_timestamp('Fim_Corrida') - unix_timestamp('Inicio_Corrida'))


df_viagens.show(5)


+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+-----------------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_Aeroporto|Taxa_Congestionamento_CBD|Mes_Corrida|Nome_Mes|Dia_Corrida|Data_Corrida|Hora_Corrida|Periodo_Dia|Dia_Semana|Numero_Dia_Semana|D

###**Tratamentos de qualidades dos dados**

###**Dados Nulos**

**Checando dados nulos:**

In [0]:
for coluna in df_viagens.columns:
    qtd_nulos = df_viagens.filter(col(coluna).isNull()).count()
    print(coluna, qtd_nulos)

Id_Fornecedor_Tecnologia 0
Inicio_Corrida 0
Fim_Corrida 0
Qtd_Passageiros 2263749
Distancia_corrida_milhas 0
Id_tarifa 2263749
Flag_armazenamento 2263749
Zona_inicio_corrida 0
Zona_fim_corrida 0
Id_tipo_pagamento 0
Valor_corrida_Tempo_Distancia 0
Taxas_Diversas 0
Taxa_MTA 0
Valor_Gorgetas_Cartao_Credito 0
Valor_Pedagios 0
Taxa_Melhoria 0
Valor_Total_Corrida 0
Taxa_Congestionamento 2263749
Taxa_Aeroporto 2263749
Taxa_Congestionamento_CBD 0
Mes_Corrida 0
Nome_Mes 0
Dia_Corrida 0
Data_Corrida 0
Hora_Corrida 0
Periodo_Dia 0
Dia_Semana 0
Numero_Dia_Semana 0
Duracao_Viagem_Segundos 0


**Análise dos valores nulos - buscando entender se são inválidos ou não:**

In [0]:
#Analisando o percentual de valores nulos:

total_registros = df_viagens.count()

for coluna in df_viagens.columns:
    qtd_nulos = df_viagens.filter(col(coluna).isNull()).count()
    percentual = (qtd_nulos / total_registros) * 100

    print( 
        f"{coluna}: "
        f"{qtd_nulos} nulos"
        f"({percentual:.2f}%)"
        )

Id_Fornecedor_Tecnologia: 0 nulos(0.00%)
Inicio_Corrida: 0 nulos(0.00%)
Fim_Corrida: 0 nulos(0.00%)
Qtd_Passageiros: 2263749 nulos(20.22%)
Distancia_corrida_milhas: 0 nulos(0.00%)
Id_tarifa: 2263749 nulos(20.22%)
Flag_armazenamento: 2263749 nulos(20.22%)
Zona_inicio_corrida: 0 nulos(0.00%)
Zona_fim_corrida: 0 nulos(0.00%)
Id_tipo_pagamento: 0 nulos(0.00%)
Valor_corrida_Tempo_Distancia: 0 nulos(0.00%)
Taxas_Diversas: 0 nulos(0.00%)
Taxa_MTA: 0 nulos(0.00%)
Valor_Gorgetas_Cartao_Credito: 0 nulos(0.00%)
Valor_Pedagios: 0 nulos(0.00%)
Taxa_Melhoria: 0 nulos(0.00%)
Valor_Total_Corrida: 0 nulos(0.00%)
Taxa_Congestionamento: 2263749 nulos(20.22%)
Taxa_Aeroporto: 2263749 nulos(20.22%)
Taxa_Congestionamento_CBD: 0 nulos(0.00%)
Mes_Corrida: 0 nulos(0.00%)
Nome_Mes: 0 nulos(0.00%)
Dia_Corrida: 0 nulos(0.00%)
Data_Corrida: 0 nulos(0.00%)
Hora_Corrida: 0 nulos(0.00%)
Periodo_Dia: 0 nulos(0.00%)
Dia_Semana: 0 nulos(0.00%)
Numero_Dia_Semana: 0 nulos(0.00%)
Duracao_Viagem_Segundos: 0 nulos(0.00%)


In [0]:
#Verificando se os valores nulos sempre aparecem juntos. Se der o valor de 2263749, estão juntos:

df_viagens.filter(col("Qtd_Passageiros").isNull()).filter(col("Id_tarifa").isNull()).filter(col("Flag_armazenamento").isNull()).filter(col("Taxa_Congestionamento").isNull()).filter(col("Taxa_Aeroporto").isNull()).count()

2263749

In [0]:
#Investigando se há algum padrão entre os valores nulos

df_viagens.filter(col("Qtd_Passageiros").isNull()).select(
    "Id_Fornecedor_Tecnologia",
    "Inicio_Corrida",
    "Fim_Corrida",
    "Zona_inicio_corrida",
    "Zona_fim_corrida",
    "Id_tipo_pagamento",
    "Valor_Total_Corrida"
).show(20, truncate=False)

+------------------------+-------------------+-------------------+-------------------+----------------+-----------------+-------------------+
|Id_Fornecedor_Tecnologia|Inicio_Corrida     |Fim_Corrida        |Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_Total_Corrida|
+------------------------+-------------------+-------------------+-------------------+----------------+-----------------+-------------------+
|2                       |2025-03-01 00:38:00|2025-03-01 00:47:00|68                 |249             |0                |4.76               |
|2                       |2025-03-01 00:12:10|2025-03-01 00:12:27|152                |152             |0                |18.95              |
|2                       |2025-03-01 00:27:50|2025-03-01 00:35:59|42                 |41              |0                |2.0                |
|2                       |2025-03-01 00:48:18|2025-03-01 01:14:38|24                 |88              |0                |43.4               |
|2    

In [0]:
#Após notar que os valores nulos sempre tinham o Id_tipo_pagamento zerado, vamos testar o que acontece com valores não nulos

df_viagens.filter(col("Qtd_Passageiros").isNotNull()).select(
    "Id_Fornecedor_Tecnologia",
    "Inicio_Corrida",
    "Fim_Corrida",
    "Zona_inicio_corrida",
    "Zona_fim_corrida",
    "Id_tipo_pagamento",
    "Valor_Total_Corrida"
).show(20, truncate=False)

+------------------------+-------------------+-------------------+-------------------+----------------+-----------------+-------------------+
|Id_Fornecedor_Tecnologia|Inicio_Corrida     |Fim_Corrida        |Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_Total_Corrida|
+------------------------+-------------------+-------------------+-------------------+----------------+-----------------+-------------------+
|1                       |2025-03-01 00:17:16|2025-03-01 00:25:52|140                |236             |1                |15.5               |
|1                       |2025-03-01 00:37:38|2025-03-01 00:43:51|140                |262             |1                |13.8               |
|2                       |2025-03-01 00:24:35|2025-03-01 00:39:49|161                |68              |1                |25.81              |
|2                       |2025-03-01 00:56:16|2025-03-01 01:01:35|231                |13              |1                |15.54              |
|1    

In [0]:
#Tirando a prova final. Agrupando por Id_tipo_pagamento as colunas que tem valores nulos para entender se acontece ou não em outros tipos de pagamento, além do de valor "0"

df_viagens.groupBy("Id_tipo_pagamento").agg(
    count("*").alias("Total_Registros"),
    count("Qtd_Passageiros").alias("Qtd._Passageiros_Preenchida"),
    count("Id_tarifa").alias("Id_tarifa_preenchido"),
    count("Flag_armazenamento").alias("Flag_Armazenamento_Preenchido"),
    count("Taxa_Congestionamento").alias("Taxa_Congestionamento_Preenchida"),
    count("Taxa_Aeroporto").alias("Taxa_Aeroporto_Preenchida")
).orderBy("Id_tipo_pagamento").show()

+-----------------+---------------+---------------------------+--------------------+-----------------------------+--------------------------------+-------------------------+
|Id_tipo_pagamento|Total_Registros|Qtd._Passageiros_Preenchida|Id_tarifa_preenchido|Flag_Armazenamento_Preenchido|Taxa_Congestionamento_Preenchida|Taxa_Aeroporto_Preenchida|
+-----------------+---------------+---------------------------+--------------------+-----------------------------+--------------------------------+-------------------------+
|                0|        2263749|                          0|                   0|                            0|                               0|                        0|
|                1|        7484734|                    7484734|             7484734|                      7484734|                         7484734|                  7484734|
|                2|        1135757|                    1135757|             1135757|                      1135757|                

Relatório de valores nulos: 

A análise e investigação de valores nulos identificou um total de 2.263.749 registros, representando 20,22%, estando presentes nas seguintes colunas: 

- Qtd_Passageiros
- Id_tarifa
- Flag_armazenamento
- Taxa_Congestionamento
- Taxa_Aeroporto

Foi possível associar os valores nulos com a coluna Id_tipo_pagamento. A análise investigativa teve como conclusão o seguinte:

- Todos os registros com Id_tipo_pagamento com valor "0" apresentam valores nulos nas cinco colunas descritas acima. 
- Todos os registros com Id_tipo_pagamento diferente do valor "0" apresentam valores preenchidos nas cinco colunas descritas acima. 
- Não foram identificados outros padrões.

Assim, optou-se por manter o padrão de valores nulos na camada Silver, não realizando substituição pelo valor "zero" ou excluindo os registros.

**Verificando duplicidades**

In [0]:
total_registros = df_viagens.count()

total_registros_distintos = df_viagens.dropDuplicates().count()

print(f"Total de Registros: {total_registros}.")
print(f"Total de Registros Únicos: {total_registros_distintos}.")
print(f"Possíveis dados duplicados: {total_registros - total_registros_distintos}")

Total de Registros: 11198026.
Total de Registros Únicos: 11198026.
Possíveis dados duplicados: 0


Após a anaálise de duplicidades realizadas na última célula, não foram encontrados registros de duplicidades exatas.

**Checando Dados Inválidos:**

In [0]:
#Quantidade de viagens com duração igual o menor a 0

df_viagens.filter(col("Duracao_Viagem_Segundos") < 0).count()

298

In [0]:
#Quantidade de viagens com duração igual a 0

df_viagens.filter(col("Duracao_Viagem_Segundos") == 0).count()

29197

In [0]:
#Investigando as viagens com duração igual a 0

df_viagens.filter(col("Duracao_Viagem_Segundos") == 0).select(
    "Inicio_Corrida",
    "Fim_Corrida",
    "Distancia_corrida_milhas",
    "Zona_inicio_corrida",
    "Zona_fim_corrida",
    "Id_tipo_pagamento",
    "Valor_Total_Corrida"
).show(20, truncate=False)

+-------------------+-------------------+------------------------+-------------------+----------------+-----------------+-------------------+
|Inicio_Corrida     |Fim_Corrida        |Distancia_corrida_milhas|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_Total_Corrida|
+-------------------+-------------------+------------------------+-------------------+----------------+-----------------+-------------------+
|2025-03-01 00:30:08|2025-03-01 00:30:08|1.33                    |234                |161             |1                |16.38              |
|2025-03-01 00:11:55|2025-03-01 00:11:55|1.54                    |163                |238             |1                |18.9               |
|2025-03-01 00:51:49|2025-03-01 00:51:49|1.34                    |114                |107             |1                |17.22              |
|2025-03-01 00:57:24|2025-03-01 00:57:24|0.95                    |107                |170             |1                |15.54              |
|2025-

In [0]:
#Quantidade de viagens com distância em milhas menor que 0
df_viagens.filter(col("Distancia_corrida_milhas") < 0).count()

0

In [0]:
#Quantidade de viagens com distância em milhas igual a 0

df_viagens.filter(col("Distancia_corrida_milhas") == 0).count()

294386

In [0]:
#Investigando as viagens com distância em milhas igual a 0

df_viagens.filter(col("Distancia_corrida_milhas") == 0).select(
    "Inicio_Corrida",
    "Fim_Corrida",
    "Distancia_corrida_milhas",
    "Zona_inicio_corrida",
    "Zona_fim_corrida",
    "Id_tipo_pagamento",
    "Valor_Total_Corrida"
).show(30, truncate=False)

+-------------------+-------------------+------------------------+-------------------+----------------+-----------------+-------------------+
|Inicio_Corrida     |Fim_Corrida        |Distancia_corrida_milhas|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_Total_Corrida|
+-------------------+-------------------+------------------------+-------------------+----------------+-----------------+-------------------+
|2025-03-01 00:51:04|2025-03-01 00:53:29|0.0                     |237                |43              |1                |14.0               |
|2025-03-01 00:02:22|2025-03-01 01:14:59|0.0                     |48                 |76              |1                |59.94              |
|2025-03-01 00:15:01|2025-03-01 00:15:17|0.0                     |232                |232             |3                |8.75               |
|2025-03-01 00:35:56|2025-03-01 00:36:00|0.0                     |164                |164             |1                |22.42              |
|2025-

In [0]:
#Quantidade de viagens com valor menor do que 0

df_viagens.filter(col("Valor_Total_Corrida") < 0).count()

186902

In [0]:
#Quantidade de viagens com valor igual a 0

df_viagens.filter(col("Valor_Total_Corrida") == 0).count()

1602

**Investigação Cruzada - Distância, Duração e Valor**

In [0]:
#Investigando Distância = 0 + Duração = 0

df_viagens.filter(col("Distancia_corrida_milhas") == 0).filter(col("Duracao_Viagem_Segundos") == 0).count()

2120

In [0]:
#Investigando Distância = 0 + Duração > 0 

df_viagens.filter(col("Distancia_corrida_milhas") == 0).filter(col("Duracao_Viagem_Segundos") > 0).count()

292266

In [0]:
#Investigando Distância = 0 + Valor Corrida = 0

df_viagens.filter(col("Distancia_corrida_milhas") == 0).filter(col("Valor_Total_Corrida") == 0).count()

768

In [0]:
#Investigando Distância = 0 + Valor Corrida > 0 

df_viagens.filter(col("Distancia_corrida_milhas") == 0).filter(col("Valor_Total_Corrida") >0).count()

280197

In [0]:
#Investigando Distância = 0 + Valor Corrida < 0

df_viagens.filter(col("Distancia_corrida_milhas") == 0).filter(col("Valor_Total_Corrida") < 0).count()

13421

In [0]:
#Verificando o comportamento da Distância = 0 + Duração da Viagem = 0 

df_viagens.filter(col("Distancia_corrida_milhas") == 0).filter(col("Duracao_Viagem_Segundos") == 0).select(
    "Inicio_Corrida",
    "Fim_Corrida",
    "Distancia_corrida_milhas",
    "Duracao_Viagem_Segundos",
    "Zona_inicio_corrida",
    "Zona_fim_corrida",
    "Id_tipo_pagamento",
    "Valor_Total_Corrida"
).show(30, truncate=False)

+-------------------+-------------------+------------------------+-----------------------+-------------------+----------------+-----------------+-------------------+
|Inicio_Corrida     |Fim_Corrida        |Distancia_corrida_milhas|Duracao_Viagem_Segundos|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_Total_Corrida|
+-------------------+-------------------+------------------------+-----------------------+-------------------+----------------+-----------------+-------------------+
|2025-03-01 00:12:13|2025-03-01 00:12:13|0.0                     |0                      |87                 |264             |2                |16.45              |
|2025-03-01 00:21:44|2025-03-01 00:21:44|0.0                     |0                      |148                |264             |2                |17.15              |
|2025-03-01 00:22:18|2025-03-01 00:22:18|0.0                     |0                      |148                |264             |2                |17.15              |
|202

In [0]:
#Investigando o comportamento do valor da corrida negativo

df_viagens.filter(col("Valor_Total_Corrida") < 0).select(
    "Inicio_Corrida",
    "Fim_Corrida",
    "Distancia_corrida_milhas",
    "Duracao_Viagem_Segundos",
    "Zona_inicio_corrida",
    "Zona_fim_corrida",
    "Id_tipo_pagamento",
    "Valor_Total_Corrida"
).show(30, truncate=False)


+-------------------+-------------------+------------------------+-----------------------+-------------------+----------------+-----------------+-------------------+
|Inicio_Corrida     |Fim_Corrida        |Distancia_corrida_milhas|Duracao_Viagem_Segundos|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_Total_Corrida|
+-------------------+-------------------+------------------------+-----------------------+-------------------+----------------+-----------------+-------------------+
|2025-03-01 00:03:53|2025-03-01 00:11:31|1.66                    |458                    |114                |68              |4                |-15.75             |
|2025-03-01 00:46:20|2025-03-01 00:46:26|0.01                    |6                      |163                |163             |2                |-74.75             |
|2025-03-01 00:07:57|2025-03-01 00:13:36|0.79                    |339                    |186                |246             |4                |-12.95             |
|202

In [0]:
#Agrupando por tipo de pagamento quando o valor total da corrida é menor que 0 

df_viagens.filter(col("Valor_Total_Corrida") < 0).groupBy("Id_tipo_pagamento").count().orderBy("Id_tipo_pagamento").show()

+-----------------+------+
|Id_tipo_pagamento| count|
+-----------------+------+
|                0|  3449|
|                1|    49|
|                2| 44415|
|                3| 23808|
|                4|115181|
+-----------------+------+



In [0]:
#Verificando anomalias no agrupamento pelo valor total da corrida

df_viagens.filter(
    col("Valor_Total_Corrida") < 0
).groupBy(
    "Valor_Total_Corrida"
).count().orderBy(
    col("Valor_Total_Corrida").desc()
).show(30, truncate=False)

+-------------------+-----+
|Valor_Total_Corrida|count|
+-------------------+-----+
|-0.01              |3    |
|-0.02              |8    |
|-0.03              |4    |
|-0.04              |8    |
|-0.05              |3    |
|-0.06              |7    |
|-0.07              |4    |
|-0.08              |1    |
|-0.09              |9    |
|-0.1               |4    |
|-0.11              |6    |
|-0.12              |8    |
|-0.13              |13   |
|-0.14              |7    |
|-0.15              |6    |
|-0.16              |8    |
|-0.17              |4    |
|-0.18              |9    |
|-0.19              |6    |
|-0.2               |7    |
|-0.21              |3    |
|-0.22              |4    |
|-0.23              |3    |
|-0.24              |5    |
|-0.25              |4    |
|-0.26              |7    |
|-0.27              |3    |
|-0.28              |4    |
|-0.29              |2    |
|-0.3               |5    |
+-------------------+-----+
only showing top 30 rows


**Criando uma flag de duração negativa da corrida**

In [0]:
df_viagens = df_viagens.withColumn("Flag_duracao_negativa", when(col("Duracao_Viagem_Segundos") < 0, 1).otherwise(0))

**Criando uma flag de duração da corrida igual a zero**

In [0]:
df_viagens = df_viagens.withColumn("Flag_duracao_zero", when(col("Duracao_Viagem_Segundos") == 0, 1).otherwise(0))

**Criando uma flag de distância negativa da corrida**

In [0]:
df_viagens = df_viagens.withColumn("Flag_distancia_negativa", when(col("Distancia_corrida_milhas") < 0, 1).otherwise(0))

**Criando uma flag de distância = 0**

In [0]:
df_viagens = df_viagens.withColumn("Flag_distancia_zero", when(col("Distancia_corrida_milhas") == 0, 1).otherwise(0))

**Criando uma flag de valor negativo da corrida**

In [0]:
df_viagens = df_viagens.withColumn("Flag_valor_negativo", when(col("Valor_Total_Corrida") < 0, 1).otherwise(0))

**Criando uma flag de valor da corrida = 0**

In [0]:

df_viagens = df_viagens.withColumn("Flag_valor_zero", when(col("Valor_Total_Corrida") ==0, 1).otherwise(0))

**Criando uma classificação geral de qualidade dos dados**

In [0]:
df_viagens = df_viagens.withColumn("Status_qualidade", 
    when(
        (col("Duracao_Viagem_Segundos") < 0) |
        (col("Valor_Total_Corrida") < 0), 
        "Invalido"
    )
    .when(
        (col("Duracao_Viagem_Segundos") == 0) |
        (col("Distancia_corrida_milhas") == 0) |
        (col("Valor_Total_Corrida") == 0),
        "Atencao"
    )
    .otherwise("Valido")
)

In [0]:
df_viagens.show()

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+-----------------------+---------------------+-----------------+-----------------------+-------------------+-------------------+---------------+----------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valor_Pedagios|Taxa_Melhoria|Valor_Total_Corrida|Taxa_Congestionamento|Taxa_

###**Decisões de tratamento de dados**

**Nulos**

Os valores nulos estão concentrados nos registros com Id_tipo_pagamento = 0. Após a análise, ficou demonstrado que não se trata de um padrão aleatório de ausência de informação. Dessa forma, os valores serão mantidos como nulos na camada Silver, preservando a informação original da fonte. 

**Duplicidades**

Não foram encontradas duplicidades exatas no conjunto de dados. Sendo assim, não houve remoção de nenhum registro por duplicidade.

**Dados com duração negativa, duração = 0, distância = 0 e valor negativo**

Foram criadas "flags" 0 e 1, sendo 1 positivo, para indicar que o dado é negativo ou de duração = 0 ou distância = 0 ou o valor da corrida é negativo


**Ao final foi criada uma coluna de status de qualidade dos dados, se dividindo em inválido, atenção e válido. Os critérios estabelecidos estão abaixo**

- Dados de duração da viagem negativos ou valor total da corrida negativo -> dados inválidos;
- Duração da viagem igual a zero ou distância da corrida igual a zero ou valor da corrida igual a zero -> deve-se ter atenção ao usar esses dados;
- Os registros restantes foram apontados como válidos.

##**Joins com a tabela de zonas**

**Preparando a tabela de zonas**


In [0]:
#Criando uma dataframe de "Zonas Início"

df_zonas_inicio = df_zonas.select(
    col("Id_Localizacao").alias("Id_zona_inicio"),
    col("Distrito").alias("Distrito_inicio"),
    col("Zona").alias("Zona_inicio"),
    col("Zona_Servico").alias("Zona_servico_inicio")    
)

df_zonas_inicio.show(10)

+--------------+---------------+--------------------+-------------------+
|Id_zona_inicio|Distrito_inicio|         Zona_inicio|Zona_servico_inicio|
+--------------+---------------+--------------------+-------------------+
|             1|            EWR|      Newark Airport|                EWR|
|             2|         Queens|         Jamaica Bay|          Boro Zone|
|             3|          Bronx|Allerton/Pelham G...|          Boro Zone|
|             4|      Manhattan|       Alphabet City|        Yellow Zone|
|             5|  Staten Island|       Arden Heights|          Boro Zone|
|             6|  Staten Island|Arrochar/Fort Wad...|          Boro Zone|
|             7|         Queens|             Astoria|          Boro Zone|
|             8|         Queens|        Astoria Park|          Boro Zone|
|             9|         Queens|          Auburndale|          Boro Zone|
|            10|         Queens|        Baisley Park|          Boro Zone|
+--------------+---------------+------

In [0]:
#Criando um dataframe de "Zonas Fim"

df_zonas_fim = df_zonas.select(
    col("Id_Localizacao").alias("Id_zona_fim"),
    col("Distrito").alias("Distrito_fim"),
    col("Zona").alias("Zona_fim"),
    col("Zona_Servico").alias("Zona_servico_fim")
)

df_zonas_fim.show(10)

+-----------+-------------+--------------------+----------------+
|Id_zona_fim| Distrito_fim|            Zona_fim|Zona_servico_fim|
+-----------+-------------+--------------------+----------------+
|          1|          EWR|      Newark Airport|             EWR|
|          2|       Queens|         Jamaica Bay|       Boro Zone|
|          3|        Bronx|Allerton/Pelham G...|       Boro Zone|
|          4|    Manhattan|       Alphabet City|     Yellow Zone|
|          5|Staten Island|       Arden Heights|       Boro Zone|
|          6|Staten Island|Arrochar/Fort Wad...|       Boro Zone|
|          7|       Queens|             Astoria|       Boro Zone|
|          8|       Queens|        Astoria Park|       Boro Zone|
|          9|       Queens|          Auburndale|       Boro Zone|
|         10|       Queens|        Baisley Park|       Boro Zone|
+-----------+-------------+--------------------+----------------+
only showing top 10 rows


**Criando o join da Zona de Origem da corrida**

In [0]:
df_viagens_enriquecida = df_viagens.join(
    broadcast(df_zonas_inicio),
    df_viagens["Zona_inicio_corrida"] == df_zonas_inicio["Id_zona_inicio"],
    "left"
)

df_viagens_enriquecida.show(10)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+-----------------------+---------------------+-----------------+-----------------------+-------------------+-------------------+---------------+----------------+--------------+---------------+--------------------+-------------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MTA|Valor_Gorgetas_Cartao_Credito|Valo

**Criando o join da Zona de Destino da corrida**

In [0]:
df_viagens_enriquecida = df_viagens_enriquecida.join(
    broadcast(df_zonas_fim),
    df_viagens_enriquecida["Zona_fim_corrida"] == df_zonas_fim["Id_zona_fim"],
    "left"
)

df_viagens_enriquecida.show(10)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+-----------------------+---------------------+-----------------+-----------------------+-------------------+-------------------+---------------+----------------+--------------+---------------+--------------------+-------------------+-----------+------------+--------------------+----------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Dista

In [0]:
df_viagens_enriquecida.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonBroadcastHashJoin LeftOuter (16)
         :- PhotonBroadcastHashJoin LeftOuter (10)
         :  :- PhotonProject (4)
         :  :  +- PhotonProject (3)
         :  :     +- PhotonProject (2)
         :  :        +- PhotonScan parquet nyc_taxi_data.bronze.viagens (1)
         :  +- PhotonShuffleExchangeSource (9)
         :     +- PhotonShuffleMapStage (8)
         :        +- PhotonShuffleExchangeSink (7)
         :           +- PhotonProject (6)
         :              +- PhotonScan parquet nyc_taxi_data.bronze.zonas (5)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonProject (12)
                     +- PhotonScan parquet nyc_taxi_data.bronze.zonas (11)


(1) PhotonScan parquet nyc_taxi_data.bronze.viagens
Output [20]: [VendorID#20398,

In [0]:
#Checando o Schema do novo dataframe

df_viagens_enriquecida.printSchema()

root
 |-- Id_Fornecedor_Tecnologia: integer (nullable = true)
 |-- Inicio_Corrida: timestamp_ntz (nullable = true)
 |-- Fim_Corrida: timestamp_ntz (nullable = true)
 |-- Qtd_Passageiros: long (nullable = true)
 |-- Distancia_corrida_milhas: double (nullable = true)
 |-- Id_tarifa: long (nullable = true)
 |-- Flag_armazenamento: string (nullable = true)
 |-- Zona_inicio_corrida: integer (nullable = true)
 |-- Zona_fim_corrida: integer (nullable = true)
 |-- Id_tipo_pagamento: long (nullable = true)
 |-- Valor_corrida_Tempo_Distancia: double (nullable = true)
 |-- Taxas_Diversas: double (nullable = true)
 |-- Taxa_MTA: double (nullable = true)
 |-- Valor_Gorgetas_Cartao_Credito: double (nullable = true)
 |-- Valor_Pedagios: double (nullable = true)
 |-- Taxa_Melhoria: double (nullable = true)
 |-- Valor_Total_Corrida: double (nullable = true)
 |-- Taxa_Congestionamento: double (nullable = true)
 |-- Taxa_Aeroporto: double (nullable = true)
 |-- Taxa_Congestionamento_CBD: double (nullable

**Análise da estratégia de Join**

A tabela de viagens contém aproximadamente 11,2 milhões de registros, enquanto a dimensão de zonas possui volume significativamente menor.

Por esse motivo, foi utilizada a estratégia de `Broadcast Join` para as dimensões de zona de origem e destino.

A análise do plano físico confirmou a utilização de:

`PhotonBroadcastHashJoin LeftOuter`

em ambos os relacionamentos.

O broadcast permite distribuir a pequena dimensão de zonas aos executores, reduzindo a necessidade de redistribuição da tabela de viagens durante o processamento.

O tipo `LEFT JOIN` foi mantido para preservar integralmente os registros da tabela principal de viagens.

Antes da realização dos joins, as colunas da dimensão foram renomeadas para diferenciar os atributos de origem e destino e evitar conflitos de nomes.

In [0]:
#Recuperando a quantidade de registro no dataframe de viagens sem os joins

df_viagens.count()

11198026

In [0]:
#Checando a quantidade de registros no novo dataframe

df_viagens_enriquecida.count()


11198026

In [0]:
#Checando se há zonas_inicio nulas

df_viagens_enriquecida.filter(
    col("Zona_inicio").isNull()
).count()

0

In [0]:
#Checando se há zonas_fim nulas

df_viagens_enriquecida.filter(
    col("Zona_fim").isNull()
).count()

0

**Relatório Joins**

Foram realizados dois left joins entre a tabela de viagens e a tabela de zonas:

- Join 1: Zona_inicio_corrida com Id_zona_inicio
- Join 2: Zona_fim_corrida com Id_zona_fim

Após os joins, o dataframe enriquecido permaneceu com o mesmo total de registros que o dataframe original de viagens, um valor de 11.198.026 registros. Validou-se também se há zonas, sendo início ou fim, com valores nulos e o resultado foi de zero zonas com valores nulos. 

Conclusão é de que os joins não provocaram duplicação de dados ou de perda de registros.

In [0]:
df_viagens_enriquecida.show(10)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+-----------------------+---------------------+-----------------+-----------------------+-------------------+-------------------+---------------+----------------+--------------+---------------+--------------------+-------------------+-----------+------------+--------------------+----------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Dista

**Selecionando as colunas do dataframe enriquecido que serão persistidas na tabela silver**

In [0]:
colunas_silver = [
    "Id_Fornecedor_Tecnologia",
    "Inicio_Corrida",
    "Fim_Corrida",
    "Qtd_Passageiros",
    "Distancia_corrida_milhas",
    "Id_tarifa",
    "Flag_armazenamento",
    "Zona_inicio_corrida",
    "Zona_fim_corrida",
    "Id_tipo_pagamento",
    "Valor_corrida_Tempo_Distancia",
    "Taxas_Diversas",
    "Taxa_MTA",
    "Valor_Gorgetas_Cartao_Credito",
    "Valor_Pedagios",
    "Taxa_Melhoria",
    "Valor_Total_Corrida",
    "Taxa_Congestionamento",
    "Taxa_Aeroporto",
    "Taxa_Congestionamento_CBD",
    "Mes_Corrida",
    "Nome_Mes",
    "Dia_Corrida",
    "Data_Corrida",
    "Hora_Corrida",
    "Periodo_Dia",
    "Dia_Semana",
    "Numero_Dia_Semana",
    "Duracao_Viagem_Segundos",
    "Flag_duracao_negativa",
    "Flag_duracao_zero",
    "Flag_distancia_negativa",
    "Flag_distancia_zero",
    "Flag_valor_negativo",
    "Flag_valor_zero",
    "Status_qualidade",
    "Distrito_inicio",
    "Zona_inicio",
    "Zona_servico_inicio",
    "Distrito_fim",
    "Zona_fim",
    "Zona_servico_fim"
]

**Criando um dataframe com a seleção de colunas do datafram enriquecido de viagens**

In [0]:
df_silver = df_viagens_enriquecida.select(colunas_silver)

df_silver.show(5)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+-----------------------+---------------------+-----------------+-----------------------+-------------------+-------------------+---------------+----------------+---------------+--------------------+-------------------+------------+--------------------+----------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MT

In [0]:
df_silver.printSchema()

root
 |-- Id_Fornecedor_Tecnologia: integer (nullable = true)
 |-- Inicio_Corrida: timestamp_ntz (nullable = true)
 |-- Fim_Corrida: timestamp_ntz (nullable = true)
 |-- Qtd_Passageiros: long (nullable = true)
 |-- Distancia_corrida_milhas: double (nullable = true)
 |-- Id_tarifa: long (nullable = true)
 |-- Flag_armazenamento: string (nullable = true)
 |-- Zona_inicio_corrida: integer (nullable = true)
 |-- Zona_fim_corrida: integer (nullable = true)
 |-- Id_tipo_pagamento: long (nullable = true)
 |-- Valor_corrida_Tempo_Distancia: double (nullable = true)
 |-- Taxas_Diversas: double (nullable = true)
 |-- Taxa_MTA: double (nullable = true)
 |-- Valor_Gorgetas_Cartao_Credito: double (nullable = true)
 |-- Valor_Pedagios: double (nullable = true)
 |-- Taxa_Melhoria: double (nullable = true)
 |-- Valor_Total_Corrida: double (nullable = true)
 |-- Taxa_Congestionamento: double (nullable = true)
 |-- Taxa_Aeroporto: double (nullable = true)
 |-- Taxa_Congestionamento_CBD: double (nullable

In [0]:
len(df_silver.columns)

42

In [0]:
df_silver.count()

11198026

**Persistindo a tabela silver**

In [0]:
tabela_silver = "nyc_taxi_data.silver.viagens"

df_silver.write.format("delta").mode("overwrite").saveAsTable(tabela_silver)

In [0]:
%sql
SELECT
    *
FROM 
    nyc_taxi_data.silver.viagens
LIMIT 5;

Id_Fornecedor_Tecnologia,Inicio_Corrida,Fim_Corrida,Qtd_Passageiros,Distancia_corrida_milhas,Id_tarifa,Flag_armazenamento,Zona_inicio_corrida,Zona_fim_corrida,Id_tipo_pagamento,Valor_corrida_Tempo_Distancia,Taxas_Diversas,Taxa_MTA,Valor_Gorgetas_Cartao_Credito,Valor_Pedagios,Taxa_Melhoria,Valor_Total_Corrida,Taxa_Congestionamento,Taxa_Aeroporto,Taxa_Congestionamento_CBD,Mes_Corrida,Nome_Mes,Dia_Corrida,Data_Corrida,Hora_Corrida,Periodo_Dia,Dia_Semana,Numero_Dia_Semana,Duracao_Viagem_Segundos,Flag_duracao_negativa,Flag_duracao_zero,Flag_distancia_negativa,Flag_distancia_zero,Flag_valor_negativo,Flag_valor_zero,Status_qualidade,Distrito_inicio,Zona_inicio,Zona_servico_inicio,Distrito_fim,Zona_fim,Zona_servico_fim
1,2025-01-01T00:18:38.000,2025-01-01T00:26:59.000,1,1.6,1,N,229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0,1,Jan,1,2025-01-01,0,Madrugada,Wednesday,4,501,0,0,0,0,0,0,Valido,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone
1,2025-01-01T00:32:40.000,2025-01-01T00:35:13.000,1,0.5,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,1,Jan,1,2025-01-01,0,Madrugada,Wednesday,4,153,0,0,0,0,0,0,Valido,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone
1,2025-01-01T00:44:04.000,2025-01-01T00:46:01.000,1,0.6,1,N,141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0,1,Jan,1,2025-01-01,0,Madrugada,Wednesday,4,117,0,0,0,0,0,0,Valido,Manhattan,Lenox Hill West,Yellow Zone,Manhattan,Lenox Hill West,Yellow Zone
2,2025-01-01T00:14:27.000,2025-01-01T00:20:01.000,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0,1,Jan,1,2025-01-01,0,Madrugada,Wednesday,4,334,0,0,0,0,0,0,Valido,Manhattan,Washington Heights South,Boro Zone,Manhattan,Washington Heights South,Boro Zone
2,2025-01-01T00:21:34.000,2025-01-01T00:25:06.000,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0,1,Jan,1,2025-01-01,0,Madrugada,Wednesday,4,212,0,0,0,0,0,0,Valido,Manhattan,Washington Heights South,Boro Zone,Manhattan,Hamilton Heights,Boro Zone


In [0]:
%sql
SELECT
    Inicio_Corrida,
    Fim_Corrida,
    Data_Corrida,
    Mes_Corrida,
    Nome_Mes,
    Id_Fornecedor_Tecnologia,
    Zona_inicio_corrida,
    Zona_fim_corrida
FROM nyc_taxi_data.silver.viagens
WHERE Nome_Mes = 'Apr';

Inicio_Corrida,Fim_Corrida,Data_Corrida,Mes_Corrida,Nome_Mes,Id_Fornecedor_Tecnologia,Zona_inicio_corrida,Zona_fim_corrida
2025-04-01T00:00:07.000,2025-04-01T00:24:28.000,2025-04-01,4,Apr,2,132,82
2025-04-01T00:00:17.000,2025-04-01T00:18:42.000,2025-04-01,4,Apr,2,43,166
